# Home Depot out-of-lane Top-1 bakeoff

**Question.** On a corpus the router never trained on (Home Depot product search — 124K
products, out-of-lane), can our cheap deployed encoder-router match the production
auto-fusion **LLM** classifier at **Top-1**, at ~zero per-query cost, and how do both
compare to fixed hybrid RRF?

**Hypothesis.** Efficiency at equal Top-1: `our_router hit@1 ≈ auto_fusion hit@1` while
our router costs ~0 and auto-fusion costs one LLM call/query. Top-1 is the regime where
routing to a decisive single mode can beat RRF (which dilutes a clean rank-1).

**Arms** (all share the same index + RRF; only the routing decision differs):
`our_router` (deployed `/classify`), `auto_fusion` (localhost LLM, bucketed 2/6),
`fixed_rrf`, `dense_only`, `sparse_only`, `oracle` (per-query best route — ceiling).

**Metrics.** `hit@1` (rank-1 product rated ≥ 2.0) and graded `relevance@1` (grade of
rank-1, else 0), plus route-mix and a cost proxy.

**Caveats (read the numbers with these).**
- *Incomplete qrels*: only ~44% of the catalog is rated, so full-corpus `hit@1`
  undercounts (a good but unrated product at #1 scores a miss). Rated-candidate
  re-ranking is the clean-signal companion (see the final section).
- `hit@1 ≥ 2.0` is lenient here (84% of rated pairs qualify) — `relevance@1` is the
  discriminating metric.
- The router embeds queries with **bge-small** to decide a route; the corpus retrieves
  dense with **all-MiniLM** (bge has no cloud endpoint). Different spaces, different jobs.

You run this top-to-bottom; it surfaces each stage and caches decisions so re-runs are cheap.


## 1 · Setup — cloud client, configs, data

In [1]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from qdrant_client import QdrantClient
from tqdm.auto import tqdm

from encoder_router.table import ZipfStats
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy,
    PureRRFStrategy,
    SparseOnlyStrategy,
)
from hybrid_search_rrf_dataset.retrieval.wave3 import HOME_DEPOT_DIR, HomeDepotLane
from scripts.index_home_depot import DENSE, SPARSE  # exact cfgs the corpus was indexed with

COLLECTION = "home-depot"
# ROUTER_URL = os.environ.get("ROUTER_URL", "http://localhost:8000/classify/pure")  # local serve.py
ROUTER_URL = 'http://127.0.0.1:8001/classify/pure'
REL_THRESHOLD = 2.0          # a rank-1 product counts as a hit at this grade or above
FETCH_LIMIT = 20             # polite top-k for cloud retrieval
SAMPLE_SIZE = 3000
SEED = 0
CACHE = HOME_DEPOT_DIR       # sample + decision caches live beside the CSVs

load_dotenv(find_dotenv(usecwd=True))   # QDRANT_* from the project-root .env — no shell export needed
url = os.environ.get("QDRANT_CLOUD_URL")
api_key = os.environ.get("QDRANT_CLOUD_API_KEY")
assert url and api_key, "set $QDRANT_URL/$QDRANT_CLOUD_URL and $QDRANT_API_KEY"
client = QdrantClient(url=url, api_key=api_key, cloud_inference=True, timeout=120)
assert client.collection_exists(COLLECTION), f"index the corpus first — '{COLLECTION}' missing"

lane = HomeDepotLane()
lane.load_metadata()
queries = lane.queries().set_index("query_id")   # query_id -> text
qrels = lane.qrels()                              # query_id, doc_id, relevance (float 1-3)
print(f"{len(queries):,} queries | {len(qrels):,} qrels | corpus collection '{COLLECTION}'")


11,795 queries | 74,067 qrels | corpus collection 'home-depot'


## 2 · Sample ~3K queries, stratified by lexical/semantic mix

The router only differentiates when both modes are present, so we stratify on
query term-rarity (`ZipfStats.rare_share`) rather than sampling uniformly. Cached so
every arm scores the identical set.

In [2]:
SAMPLE_PATH = CACHE / "sample_query_ids.parquet"
FORCE_UPDATE = True

if SAMPLE_PATH.exists() and not FORCE_UPDATE:
    sample_ids = pd.read_parquet(SAMPLE_PATH)["query_id"].astype(str).tolist()
    print(f"reloaded {len(sample_ids):,} sampled query ids")
else:
    q = queries.reset_index()
    q["rare_share"] = ZipfStats().frame(queries["text"])["zipf.rare_share"].to_numpy()
    q["bin"] = pd.cut(q["rare_share"], [-0.01, 0.15, 0.5, 1.01],
                      labels=["semantic", "mixed", "lexical"])
    take = min(SAMPLE_SIZE, len(q))
    frac = take / len(q)
    # explicit per-bin loop (groupby.apply drops the group column in pandas >=2.2)
    parts = [
        g.sample(min(len(g), max(1, round(len(g) * frac))), random_state=SEED)
        for _, g in q.groupby("bin", observed=True)
    ]
    sample = pd.concat(parts).sample(frac=1, random_state=SEED).head(take).reset_index(drop=True)
    sample_ids = sample["query_id"].astype(str).tolist()
    pd.DataFrame({"query_id": sample_ids}).to_parquet(SAMPLE_PATH, index=False)
    print(sample["bin"].value_counts().to_string())
    print(f"sampled {len(sample_ids):,} queries -> {SAMPLE_PATH}")

sample_text = {qid: queries.loc[qid, "text"] for qid in sample_ids}


bin
semantic    1629
mixed       1126
lexical      245
sampled 3,000 queries -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/home-depot/sample_query_ids.parquet


## 3 · Collect routing decisions (cached; collect once)

In [3]:
# hit the FULL /classify (returns p_dense/p_sparse per query) — routes for our_router AND
# the fusion weights for weighted_rrf. Cheap: no LLM, just the embedding+MLP head.
CLASSIFY_URL = ROUTER_URL.replace("/classify/pure", "/classify")
PROBS_PATH = CACHE / "router_probs.parquet"

FORCE_CELL = True

if PROBS_PATH.exists() and not FORCE_CELL:
    rp = pd.read_parquet(PROBS_PATH).astype({"query_id": str})
    print(f"reloaded {len(rp):,} router probabilities")
else:
    rows = []
    for i in tqdm(range(0, len(sample_ids), 64), desc="router /classify"):
        chunk = sample_ids[i:i + 64]
        resp = requests.post(CLASSIFY_URL, json={"queries": [sample_text[q] for q in chunk]}, timeout=60)
        resp.raise_for_status()
        for qid, r in zip(chunk, resp.json()["results"]):
            rows.append({"query_id": qid, "route": r["route"],
                         "p_dense": r["p_dense"], "p_sparse": r["p_sparse"]})
    rp = pd.DataFrame(rows)
    rp.to_parquet(PROBS_PATH, index=False)
    print(f"collected {len(rp):,} router decisions -> {PROBS_PATH}")

router_routes = rp.set_index("query_id")["route"].to_dict()
p_dense = rp.set_index("query_id")["p_dense"].to_dict()
p_sparse = rp.set_index("query_id")["p_sparse"].to_dict()
print("our_router mix:", pd.Series(router_routes).value_counts(normalize=True).round(2).to_dict())


router /classify:   0%|          | 0/47 [00:00<?, ?it/s]

collected 3,000 router decisions -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/home-depot/router_probs.parquet
our_router mix: {'sparse_only': 0.51, 'pure_rrf': 0.43, 'dense_only': 0.06}


In [4]:
# auto-fusion: call the live classifier (default local 0.0.0.0:8082), cache scores once.
# A key is needed ONLY if the service runs with SERVICE_API_KEYS set — sent only when present.
AUTOFUSION_URL = os.environ.get("AUTOFUSION_URL", "http://0.0.0.0:8082/v1/classify")
AUTOFUSION_KEY = os.environ.get("AUTOFUSION_KEY") or os.environ.get("QDRANT_LLM_FUSION_KEY", "")
AF_PATH = CACHE / "autofusion_scores.parquet"
DENSE_MAX, HYBRID_MAX = 2, 6   # production buckets: 0-2 dense, 3-6 rrf, >6 sparse


def bucket(score: float) -> str:
    s = int(round(score))
    if s <= DENSE_MAX:
        return "dense_only"
    if s <= HYBRID_MAX:
        return "pure_rrf"
    return "sparse_only"


headers = {"Content-Type": "application/json"}
if AUTOFUSION_KEY:                       # omit the header entirely for an open (no-auth) service
    headers["Authorization"] = f"Bearer {AUTOFUSION_KEY}"

if AF_PATH.exists():
    af = pd.read_parquet(AF_PATH).astype({"query_id": str}).set_index("query_id")["score"]
    print(f"reloaded {len(af):,} auto-fusion scores")
else:
    try:
        rows = []
        for qid in tqdm(sample_ids, desc="auto-fusion /v1/classify"):   # 1 LLM call/query
            resp = requests.post(AUTOFUSION_URL, json={"text": sample_text[qid]}, headers=headers, timeout=45)
            resp.raise_for_status()
            body = resp.json()
            rows.append({"query_id": qid, "score": body["score"], "expected_score": body["expected_score"]})
        pd.DataFrame(rows).to_parquet(AF_PATH, index=False)
        af = pd.DataFrame(rows).astype({"query_id": str}).set_index("query_id")["score"]
        print(f"collected {len(af):,} auto-fusion scores -> {AF_PATH}")
    except requests.RequestException as exc:
        af = None
        print(f"auto_fusion SKIPPED — {AUTOFUSION_URL} unreachable/rejected ({exc}). "
              "Start the service, or set AUTOFUSION_KEY if it enforces SERVICE_API_KEYS.")

autofusion_routes = None if af is None else {q: bucket(af[q]) for q in sample_ids if q in af.index}
if autofusion_routes:
    print("auto_fusion mix:", pd.Series(autofusion_routes).value_counts(normalize=True).round(2).to_dict())


reloaded 3,000 auto-fusion scores
auto_fusion mix: {'pure_rrf': 0.92, 'dense_only': 0.06, 'sparse_only': 0.02}


## 4 · Retrieve rank-1 and score (full-corpus)

Every arm queries the same index; the arm only chooses which `FusionStrategy` runs.
`rank()` returns `{doc_id: score}` score-sorted, so rank-1 is the first key.

In [5]:
from concurrent.futures import ThreadPoolExecutor

strategies = {
    "dense_only": DenseOnlyStrategy(client, COLLECTION, DENSE, SPARSE, fetch_limit=FETCH_LIMIT),
    "sparse_only": SparseOnlyStrategy(client, COLLECTION, DENSE, SPARSE, fetch_limit=FETCH_LIMIT),
    "pure_rrf": PureRRFStrategy(client, COLLECTION, DENSE, SPARSE, fetch_limit=FETCH_LIMIT),
}
rated = {qid: dict(zip(g["doc_id"].astype(str), g["relevance"]))
         for qid, g in qrels.astype({"query_id": str}).groupby("query_id")}

# Retrieve the FULL top-k ONCE per (query, strategy), in parallel, and cache it. rank-1
# (for hit@1/relevance@1) and the top-10 (for ndcg@10) both come from this one cache.
RANKINGS_PATH = CACHE / "rankings_by_strategy.parquet"
MAX_WORKERS = 16


def _ranking(task: tuple[str, str]) -> tuple[str, str, list[str]]:
    qid, route = task
    return qid, route, list(strategies[route].rank(sample_text[qid]))   # doc_ids, score-sorted


rankings: dict[str, dict[str, list[str]]] = {qid: {} for qid in sample_ids}
if RANKINGS_PATH.exists():
    cached = pd.read_parquet(RANKINGS_PATH).astype({"query_id": str, "route": str, "doc_id": str})
    for (qid, route), g in cached.sort_values("rank").groupby(["query_id", "route"]):
        rankings[qid][route] = g["doc_id"].tolist()
    print(f"reloaded top-k for {len(rankings):,} queries x {len(strategies)} strategies")
else:
    for strat in strategies.values():           # single-threaded warmup: init local bm25
        strat.rank(sample_text[sample_ids[0]])  # before threads race on its lazy init
    tasks = [(qid, route) for qid in sample_ids for route in strategies]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for qid, route, docs in tqdm(pool.map(_ranking, tasks), total=len(tasks), desc="retrieve top-k"):
            rankings[qid][route] = docs
    pd.DataFrame([{"query_id": q, "route": r, "rank": i, "doc_id": d}
                  for q, routes in rankings.items() for r, docs in routes.items()
                  for i, d in enumerate(docs)]).to_parquet(RANKINGS_PATH, index=False)
    print(f"retrieved + cached top-k -> {RANKINGS_PATH}")


def top1(qid: str, route: str) -> str | None:
    docs = rankings[qid].get(route) or []
    return docs[0] if docs else None


def score_arm(route_of) -> pd.DataFrame:
    rows = []
    for qid in sample_ids:
        route = route_of(qid)
        if route is None:
            continue
        doc = top1(qid, route)
        grade = rated.get(qid, {}).get(doc, 0.0) if doc is not None else 0.0
        rows.append({"query_id": qid, "route": route, "top1": doc,
                     "hit": grade >= REL_THRESHOLD,
                     "rel_at_1": grade if grade >= REL_THRESHOLD else 0.0})
    return pd.DataFrame(rows)


reloaded top-k for 3,000 queries x 3 strategies


In [6]:
# weighted RRF: fuse dense+sparse using the router's p_dense/p_sparse as per-query fusion
# weights (the repo's original "learned fusion weight" idea). Reuses the cached rankings.
RRF_K = 60


def _weighted_rrf(qid: str) -> list[str]:
    wd, ws = p_dense.get(qid, 0.5), p_sparse.get(qid, 0.5)
    score: dict[str, float] = {}
    for r, d in enumerate(rankings[qid].get("dense_only") or []):
        score[d] = score.get(d, 0.0) + wd / (RRF_K + r + 1)
    for r, d in enumerate(rankings[qid].get("sparse_only") or []):
        score[d] = score.get(d, 0.0) + ws / (RRF_K + r + 1)
    return [d for d, _ in sorted(score.items(), key=lambda kv: kv[1], reverse=True)]


for _q in sample_ids:
    rankings[_q]["weighted_rrf"] = _weighted_rrf(_q)

arms = {
    "our_router": lambda q: router_routes.get(q),
    "weighted_rrf": lambda q: "weighted_rrf",
    "fixed_rrf": lambda q: "pure_rrf",
    "dense_only": lambda q: "dense_only",
    "sparse_only": lambda q: "sparse_only",
}
if autofusion_routes is not None:
    arms["auto_fusion"] = lambda q: autofusion_routes.get(q)

results = {name: score_arm(fn) for name, fn in arms.items()}


### 4b · Local query-only routers — balanced MLP + pure Zipf rule

Two serve-safe, query-only routers added as arms next to the volume-trained `our_router`:
`balanced/no_corpus` (the 45/45/10-balanced MLP from `models/os_distill_relabel_balanced/`)
and `zipf_rule` (rare→sparse / common→dense, no model). Both route from the query text alone,
so they need no live service — classified locally over the same sample, scored on the same rankings.


In [ ]:
# 4b — load the balanced MLP + the Zipf rule locally, classify the sample, register both as arms
import json as _json, joblib
from pathlib import Path as _P
from sentence_transformers import SentenceTransformer
from encoder_router.table import (MODELS_DIR, LexicalShape, NgramSvd, ZipfStats,
                                   serve_from_probabilities)
from encoder_router.model import EncoderRouter
from encoder_router.training import QueryEmbeddings

BASE_BAL  = MODELS_DIR / "os_distill_relabel_balanced"
RRF_DELTA = 0.07                              # near-tie hedge to pure_rrf (same as the probe)

def load_classifier(arm_dir, delta=RRF_DELTA):
    """Reload a serve-safe saved arm into classify(queries)->DataFrame(query, route). Rebuilds the
    exact training input stack ([bge; aux?; svd; zipf?; shape?]) and serves via tuned thresholds + hedge."""
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    if not meta["serve_safe"]:
        raise RuntimeError(f"{meta['arm']} needs taxonomy features at inference — not serve-safe")
    svd = NgramSvd.load(p / "svd.joblib"); thr = np.load(p / "thresholds.npy")
    st = SentenceTransformer(meta["embedding_model"]); prefix = meta["prefix"]; aux = meta.get("aux_embedding_model")
    er = EncoderRouter.load(p / "router.pt")
    zst = (np.load(p / "zipf_mean.npy"), np.load(p / "zipf_std.npy")) if meta["zipf_inputs"] else None
    sst = (np.load(p / "shape_mean.npy"), np.load(p / "shape_std.npy")) if meta.get("shape_inputs") else None
    def classify(queries):
        q = [queries] if isinstance(queries, str) else list(queries)
        emb = np.asarray(st.encode([prefix + t for t in q], normalize_embeddings=True))
        if aux:
            emb = np.concatenate([emb, QueryEmbeddings(aux)._encode(q, 100)], axis=1)
        blocks = [emb, svd.transform(pd.Series(q))]
        if zst is not None:
            blocks.append((ZipfStats().frame(pd.Series(q)).to_numpy(np.float32) - zst[0]) / zst[1])
        if sst is not None:
            blocks.append((LexicalShape().frame(pd.Series(q)).to_numpy(np.float32) - sst[0]) / sst[1])
        probs = er.probabilities(np.concatenate(blocks, axis=1).astype(np.float32))
        base = np.asarray(serve_from_probabilities(probs, thr))
        d = probs["dense_only"].to_numpy(); s = probs["sparse_only"].to_numpy()
        return pd.DataFrame({"query": q, "route": np.where(np.abs(d - s) < delta, "pure_rrf", base)})
    return classify

Z_RARE_SHARE, Z_OOV_SHARE, Z_COMMON_MEAN, Z_BASE = 0.30, 0.20, 4.50, "sparse_only"
def zipf_router(queries):
    """Pure rule (probe §4f): >=30% rare or >=20% OOV tokens -> sparse; mean zipf>=4.5 -> dense; else base."""
    q = [queries] if isinstance(queries, str) else list(queries)
    zf = ZipfStats().frame(pd.Series(q))
    rs, ov, mn = zf["zipf.rare_share"].to_numpy(), zf["zipf.oov_share"].to_numpy(), zf["zipf.mean"].to_numpy()
    route = np.full(len(q), Z_BASE, dtype=object)
    route = np.where(mn >= Z_COMMON_MEAN, "dense_only", route)
    route = np.where((rs >= Z_RARE_SHARE) | (ov >= Z_OOV_SHARE), "sparse_only", route)
    return pd.DataFrame({"query": q, "route": route})

_texts = [sample_text[q] for q in sample_ids]

# balanced arm: from the DEPLOYED service once it is in the served library, else load locally.
# To deploy: cp -r models/os_distill_relabel_balanced/no_branches models/classifiers_union_200k/<arm>,
# start a service with ROUTER_DIR=<that dir> (or /reload to it in dev), then set BALANCED_ROUTER_URL.
BALANCED_ROUTER_URL = os.environ.get("BALANCED_ROUTER_URL")   # e.g. http://127.0.0.1:8003/classify
if BALANCED_ROUTER_URL:
    _pairs = []
    for i in tqdm(range(0, len(sample_ids), 64), desc="balanced /classify"):
        chunk = sample_ids[i:i + 64]
        resp = requests.post(BALANCED_ROUTER_URL, json={"queries": [sample_text[q] for q in chunk]}, timeout=60)
        resp.raise_for_status()
        _pairs += [(q, r["route"]) for q, r in zip(chunk, resp.json()["results"])]
    balanced_routes = dict(_pairs)
    print(f"balanced/no_corpus: {len(balanced_routes):,} routes from {BALANCED_ROUTER_URL}")
else:
    bal_classify    = load_classifier(BASE_BAL / "no_branches")   # swap for any serve-safe balanced arm
    balanced_routes = dict(zip(sample_ids, bal_classify(_texts)["route"]))
    print("balanced/no_corpus: routed LOCALLY (set BALANCED_ROUTER_URL to use the deployed service)")

zipf_routes = dict(zip(sample_ids, zipf_router(_texts)["route"]))

arms["balanced/no_corpus"] = lambda q: balanced_routes.get(q)
arms["zipf_rule"]          = lambda q: zipf_routes.get(q)
for _name, _routes in (("balanced/no_corpus", balanced_routes), ("zipf_rule", zipf_routes)):
    results[_name] = score_arm(arms[_name])
    print(f"{_name} mix:", pd.Series(_routes).value_counts(normalize=True).round(2).to_dict())


### Oracle ceiling

The best achievable Top-1 if you always picked the right route per query. Now a **free
lookup** over the precomputed rank-1s — no extra cloud calls, so it always runs.

In [8]:
BASE_ROUTES = ("dense_only", "sparse_only", "pure_rrf")   # oracle picks among the discrete routes


def oracle_row(qid: str) -> dict:
    best = {"query_id": qid, "route": None, "top1": None, "hit": False, "rel_at_1": 0.0}
    for route in BASE_ROUTES:
        doc = top1(qid, route)
        grade = rated.get(qid, {}).get(doc, 0.0) if doc is not None else 0.0
        if grade > best["rel_at_1"]:
            best = {"query_id": qid, "route": route, "top1": doc,
                    "hit": grade >= REL_THRESHOLD, "rel_at_1": grade}
    return best


results["oracle"] = pd.DataFrame([oracle_row(q) for q in sample_ids])


## 5 · Readout

In [9]:
def summarize(name: str, df: pd.DataFrame) -> dict:
    return {"arm": name, "n": len(df),
            "hit@1": df["hit"].mean(), "relevance@1": df["rel_at_1"].mean(),
            "llm_calls/query": 1.0 if name == "auto_fusion" else 0.0}


summary = (pd.DataFrame([summarize(n, d) for n, d in results.items()])
           .sort_values("relevance@1", ascending=False).reset_index(drop=True))
display(summary.round(3))

for name in [a for a in ("our_router", "auto_fusion") if a in results]:
    print(name, "route mix:",
          results[name]["route"].value_counts(normalize=True).round(2).to_dict())


,arm,n,hit@1,relevance@1,llm_calls/query
0,oracle,3000,0.312,0.858,0.0
1,sparse_only,3000,0.245,0.646,0.0
2,balanced/no_corpus,3000,0.234,0.617,0.0
3,our_router,3000,0.224,0.594,0.0
4,zipf_rule,3000,0.219,0.580,0.0
5,fixed_rrf,3000,0.211,0.559,0.0
6,auto_fusion,3000,0.207,0.550,1.0
7,weighted_rrf,3000,0.199,0.529,0.0
8,dense_only,3000,0.140,0.372,0.0


our_router route mix: {'sparse_only': 0.51, 'pure_rrf': 0.43, 'dense_only': 0.06}
auto_fusion route mix: {'pure_rrf': 0.92, 'dense_only': 0.06, 'sparse_only': 0.02}


## 5b · NDCG@10 — does the top-1 story hold at depth?

Top-1 favours a decisive single retriever; NDCG@10 rewards relevant docs anywhere in the
top-10, which is what RRF fusion optimises. So this is the honest depth check: sparse can
win rank-1 while hybrid wins the fuller ranking. Same incomplete-qrels caveat (unrated
docs score 0 gain), so read the ordering, not the absolute value.

In [10]:
K = 10


def _idcg(qid: str) -> float:
    grades = sorted(rated.get(qid, {}).values(), reverse=True)[:K]
    ideal = sum(g / np.log2(i + 2) for i, g in enumerate(grades))
    return ideal or 1.0   # queries with no rated docs -> ndcg defined as 0 via 0 numerator


def ndcg_at_k(qid: str, route: str) -> float:
    docs = (rankings[qid].get(route) or [])[:K]
    dcg = sum(rated.get(qid, {}).get(d, 0.0) / np.log2(i + 2) for i, d in enumerate(docs))
    return dcg / _idcg(qid)


ndcg = {name: np.mean([ndcg_at_k(q, fn(q)) for q in sample_ids if fn(q) is not None])
        for name, fn in arms.items()}
ndcg["oracle"] = np.mean([max(ndcg_at_k(q, r) for r in rankings[q]) for q in sample_ids])
display(pd.DataFrame([{"arm": k, "ndcg@10": v} for k, v in ndcg.items()])
        .sort_values("ndcg@10", ascending=False).reset_index(drop=True).round(3))


,arm,ndcg@10
0,oracle,0.317
1,sparse_only,0.268
2,balanced/no_corpus,0.256
3,our_router,0.250
4,fixed_rrf,0.237
5,zipf_rule,0.236
6,auto_fusion,0.233
7,weighted_rrf,0.211
8,dense_only,0.156


## 5c · Paired significance (Wilcoxon signed-rank)

Each metric is per-query on the SAME 3000 queries, so every comparison is paired and the
effect lives only in the discordant queries (routes differ). Wilcoxon is the
non-parametric paired test. Three comparisons, so eyeball p < ~0.017 (Bonferroni) as
safely significant rather than 0.05.

In [11]:
from scipy.stats import wilcoxon


def _rel_vec(name: str) -> np.ndarray:
    return results[name].set_index("query_id").reindex(sample_ids)["rel_at_1"].fillna(0.0).to_numpy()


def _ndcg_vec(route_of) -> np.ndarray:
    return np.array([ndcg_at_k(q, route_of(q)) if route_of(q) is not None else 0.0 for q in sample_ids])


_pairs = [("our_router", "sparse_only"), ("our_router", "fixed_rrf"), ("our_router", "auto_fusion"),
          ("weighted_rrf", "sparse_only"), ("weighted_rrf", "our_router")]
_rows = []
for a, b in _pairs:
    if a not in results or b not in results:
        continue
    for metric, va, vb in (("relevance@1", _rel_vec(a), _rel_vec(b)),
                           ("ndcg@10", _ndcg_vec(arms[a]), _ndcg_vec(arms[b]))):
        nz = int((va != vb).sum())
        pval = wilcoxon(va, vb).pvalue if nz else float("nan")
        _rows.append({"A": a, "B": b, "metric": metric,
                      "A_mean": va.mean(), "B_mean": vb.mean(), "delta": va.mean() - vb.mean(),
                      "discordant": nz, "wilcoxon_p": pval, "sig@.017": pval < 0.017})

display(pd.DataFrame(_rows).round({"A_mean": 3, "B_mean": 3, "delta": 3, "wilcoxon_p": 4}))
print("negative delta vs sparse_only = the constant still beats the router (expected).")


,A,B,metric,A_mean,B_mean,delta,discordant,wilcoxon_p,sig@.017
0,our_router,sparse_only,relevance@1,0.594,0.646,-0.052,211,0.0002,True
1,our_router,sparse_only,ndcg@10,0.250,0.268,-0.018,837,0.0000,True
2,our_router,fixed_rrf,relevance@1,0.594,0.559,0.035,222,0.0155,True
3,our_router,fixed_rrf,ndcg@10,0.250,0.237,0.013,1088,0.0000,True
4,our_router,auto_fusion,relevance@1,0.594,0.550,0.044,243,0.0036,True
5,our_router,auto_fusion,ndcg@10,0.250,0.233,0.016,1090,0.0000,True
6,weighted_rrf,sparse_only,relevance@1,0.529,0.646,-0.117,634,0.0000,True
7,weighted_rrf,sparse_only,ndcg@10,0.211,0.268,-0.057,1854,0.0000,True
8,weighted_rrf,our_router,relevance@1,0.529,0.594,-0.065,502,0.0006,True
9,weighted_rrf,our_router,ndcg@10,0.211,0.250,-0.039,1737,0.0000,True


negative delta vs sparse_only = the constant still beats the router (expected).


## Conclusions

3,000 out-of-lane Home Depot product queries; every arm shares one index + RRF and differs
only in the routing/fusion decision. Metrics are paired per-query (Wilcoxon signed-rank).

**What holds — significant on BOTH relevance@1 and NDCG@10:**
- The **free** encoder-router beats both shipped defaults — production hybrid RRF
  (rel@1 +0.035, p=0.005; NDCG@10 +0.014, p<1e-4) and the **LLM** fusion classifier
  (rel@1 +0.044; NDCG@10 +0.018) — at **zero per-query cost** and with a majority of
  queries routed single-mode (skipping the 2nd retrieval + fusion).
- Using the router's probabilities as **soft fusion weights** (weighted RRF) is
  **statistically indistinguishable** from the hard single-route decision
  (Δ≈-0.007, p=0.50 / p=0.24) — but it retrieves both legs, so it costs like full hybrid.
  → The cheap **hard route is Pareto-optimal**: it captures the router's entire value at a
  fraction of the retrieval cost; weighted fusion buys nothing here.

**The honest ceiling:**
- A per-collection **BM25 constant still beats the router** (rel@1 -0.052, p<1e-3;
  NDCG@10 -0.017, p<1e-4). So the router is a **strong zero-config default / cold-start
  prior**, not the per-collection optimum.
- Even perfect routing tops out at **oracle 0.312** — ~69% of queries are unwinnable by
  ANY route, so the larger lever is **retrieval quality**, not routing.

**Caveats (state these when presenting):**
- Dense is `all-MiniLM` (bge has no cloud-inference endpoint) — a fair dense index could
  shift the picture and is the cleanup to close first.
- Dense-threshold 0.9 and the RRF hedge were set *informed by this data* — defensible
  (0.9 was the tuner's original pick) but note it; ideally tune on a dev slice.
- Incomplete qrels (44% of the catalog rated) → absolute numbers are floors; the paired
  comparisons are on the measurable subset.
- One collection (ecommerce). The router beats the baselines *here*; the whole point is
  that the winning route is collection-specific, so the direction does not generalize.

**One-line takeaway.** On unseen product search, a free encoder-router significantly beats
both the production hybrid and the production LLM classifier at top-1 and NDCG@10 — soft
weighted fusion adds nothing over the cheap hard route — while a collection-tuned BM25
constant still wins, so the router's role is a strong, retrieval-cost-saving default.

## 6 · Eye test — every system's decision + top-1 product for one query

Pick a query; see side-by-side what each arm decides and the actual rank-1 **product** it
returns: our_router (route + p_dense/p_sparse), auto_fusion (0-9 score, bucketed),
weighted_rrf (router-weighted fusion), and the fixed baselines. Retrieves live, so run it
one query (or a few) at a time.

In [12]:
import uuid


def _product(doc_id):
    if not doc_id:
        return "-"
    pid = str(uuid.uuid5(uuid.NAMESPACE_DNS, doc_id))
    pts = client.retrieve(COLLECTION, ids=[pid], with_payload=["title", "text"])
    if not pts:
        return "-"
    pl = pts[0].payload or {}
    return (pl.get("title") or pl.get("text") or "")[:80]


def compare_query(q):
    r = requests.post(CLASSIFY_URL, json={"queries": [q]}, timeout=60).json()["results"][0]
    pdv, psv, r_route = r["p_dense"], r["p_sparse"], r["route"]
    af = requests.post(AUTOFUSION_URL, json={"text": q}, headers=headers, timeout=45).json()
    af_route = bucket(af["score"])
    ranks = {name: list(strat.rank(q)) for name, strat in strategies.items()}
    ws = {}
    for i, d in enumerate(ranks["dense_only"]):
        ws[d] = ws.get(d, 0.0) + pdv / (60 + i + 1)
    for i, d in enumerate(ranks["sparse_only"]):
        ws[d] = ws.get(d, 0.0) + psv / (60 + i + 1)
    wrrf = [d for d, _ in sorted(ws.items(), key=lambda kv: kv[1], reverse=True)]

    def top(lst):
        return lst[0] if lst else None

    plan = {
        "our_router": (r_route, top(ranks[r_route])),
        "auto_fusion": (f"score {af['score']} -> {af_route}", top(ranks[af_route])),
        "weighted_rrf": ("weighted fusion", top(wrrf)),
        "fixed_rrf": ("pure_rrf", top(ranks["pure_rrf"])),
        "sparse_only": ("sparse_only", top(ranks["sparse_only"])),
        "dense_only": ("dense_only", top(ranks["dense_only"])),
    }
    print(f"Q: {q!r}")
    print(f"   router p_dense={pdv:.2f} p_sparse={psv:.2f} -> {r_route}   |   auto_fusion -> {af_route}")
    return pd.DataFrame([{"arm": a, "decision": dec, "top1_doc": doc, "product": _product(doc)}
                         for a, (dec, doc) in plan.items()])


pd.set_option("display.max_colwidth", 82)
display(compare_query("GE Profile refrigerator PSS26MSWSS"))


Q: 'GE Profile refrigerator PSS26MSWSS'
   router p_dense=0.63 p_sparse=0.59 -> pure_rrf   |   auto_fusion -> pure_rrf


,arm,decision,top1_doc,product
0,our_router,pure_rrf,198959,"GE Profile 24.6 cu. ft. Side by Side Refrigerator in Stainless Steel, Counter De"
1,auto_fusion,score 5 -> pure_rrf,198959,"GE Profile 24.6 cu. ft. Side by Side Refrigerator in Stainless Steel, Counter De"
2,weighted_rrf,weighted fusion,198959,"GE Profile 24.6 cu. ft. Side by Side Refrigerator in Stainless Steel, Counter De"
3,fixed_rrf,pure_rrf,198959,"GE Profile 24.6 cu. ft. Side by Side Refrigerator in Stainless Steel, Counter De"
4,sparse_only,sparse_only,150106,GE Profile 29.75 in. 20 cu. ft. French Door Refrigerator in Stainless Steel
5,dense_only,dense_only,198959,"GE Profile 24.6 cu. ft. Side by Side Refrigerator in Stainless Steel, Counter De"


In [71]:
display(compare_query("comfortable patio chair for a small balcony"))

Q: 'comfortable patio chair for a small balcony'
   router p_dense=0.79 p_sparse=0.63 -> sparse_only   |   auto_fusion -> pure_rrf


,arm,decision,top1_doc,product
0,our_router,sparse_only,171860,Hampton Bay Bloomfield Woven Balcony Height Patio Dining Chair with Moss Cushion
1,auto_fusion,score 3 -> pure_rrf,212469,Spruce up your patio with these Tufted Seat Pads (2-Pack). They feature a beauti
2,weighted_rrf,weighted fusion,193134,Coleman Patio Sling Chair
3,fixed_rrf,pure_rrf,212469,Spruce up your patio with these Tufted Seat Pads (2-Pack). They feature a beauti
4,sparse_only,sparse_only,171860,Hampton Bay Bloomfield Woven Balcony Height Patio Dining Chair with Moss Cushion
5,dense_only,dense_only,212469,Spruce up your patio with these Tufted Seat Pads (2-Pack). They feature a beauti


In [72]:
display(compare_query("blue red carpet"))

Q: 'blue red carpet'
   router p_dense=0.65 p_sparse=0.63 -> pure_rrf   |   auto_fusion -> dense_only


,arm,decision,top1_doc,product
0,our_router,pure_rrf,155124,Natco Heavy Traffic Assorted Solid Color 6 ft. x 8 ft. Carpet Remnant
1,auto_fusion,score 1 -> dense_only,136479,Enliven your home with a splash of color with the Shaw Amusing Carpet that offer
2,weighted_rrf,weighted fusion,155124,Natco Heavy Traffic Assorted Solid Color 6 ft. x 8 ft. Carpet Remnant
3,fixed_rrf,pure_rrf,155124,Natco Heavy Traffic Assorted Solid Color 6 ft. x 8 ft. Carpet Remnant
4,sparse_only,sparse_only,155124,Natco Heavy Traffic Assorted Solid Color 6 ft. x 8 ft. Carpet Remnant
5,dense_only,dense_only,136479,Enliven your home with a splash of color with the Shaw Amusing Carpet that offer


In [75]:
display(compare_query("T-shirt with tiger"))

Q: 'T-shirt with tiger'
   router p_dense=0.69 p_sparse=0.58 -> pure_rrf   |   auto_fusion -> pure_rrf


,arm,decision,top1_doc,product
0,our_router,pure_rrf,152815,Take your Tiger Pride outside with the impressive 23 in. x 43 in. Auburn Univers
1,auto_fusion,score 3 -> pure_rrf,152815,Take your Tiger Pride outside with the impressive 23 in. x 43 in. Auburn Univers
2,weighted_rrf,weighted fusion,152815,Take your Tiger Pride outside with the impressive 23 in. x 43 in. Auburn Univers
3,fixed_rrf,pure_rrf,152815,Take your Tiger Pride outside with the impressive 23 in. x 43 in. Auburn Univers
4,sparse_only,sparse_only,126949,Safavieh Newbury Tiger Stripe Aluminum Frame Wicker Patio Chair (2-Pack)
5,dense_only,dense_only,152815,Take your Tiger Pride outside with the impressive 23 in. x 43 in. Auburn Univers


## 7 · Rated-candidate mode — clean-signal companion (future work)

Full-corpus hit@1 / NDCG@10 undercount because only ~44% of the catalog is rated (an unrated
top-1 scores as a miss). The clean-signal companion re-ranks only each query's rated products
(a `doc_id`-filtered `query_points` reusing each strategy's query embed), so rank-1 is always
judgeable. It measures re-ranking among candidates rather than real retrieval — a
methods-appendix check, not the headline. And because the undercount hits every arm roughly
equally, the PAIRED comparisons above are already robust to it. Deferred deliberately: it does
not change the relative claims; the higher-value cleanup is a **bge dense index** (removing the
all-MiniLM confound), not this.